# Middleware

[Middleware](https://docs.langchain.com/oss/python/langchain/middleware) "provides a
way to more tightly control what happens inside the agent" — logging, prompt
rewriting, retries, approval gates.

We build one agent with a web-search tool and a SQL tool, then wrap it.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## The tools

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

## The agent, before any middleware

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
)

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]}
)

print(response["messages"][-1].content)

## The hooks

Middleware attaches to points in the agent loop. There are two shapes:

| Hook | Runs | Shape |
| --- | --- | --- |
| `before_agent` | once, when the run starts | node |
| `before_model` | before **every** model call | node |
| `after_model` | after **every** model call | node |
| `after_agent` | once, when the run ends | node |
| `wrap_model_call` | around each model call | wrap |
| `wrap_tool_call` | around each tool call | wrap |

Node-style hooks take `(state, runtime)` and return state updates or `None`.
Wrap-style hooks take `(request, handler)` and must call `handler(request)`.

See [custom middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom).

## Node-style hooks

In [ ]:
from typing import Any

from langchain.agents import AgentState
from langchain.agents.middleware import (
    after_agent,
    after_model,
    before_agent,
    before_model,
)
from langgraph.runtime import Runtime


@before_agent
def log_before_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[before_agent] run starting with {len(state['messages'])} message(s)")
    return None


@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[before_model]  {len(state['messages'])} message(s) going to the model")
    return None


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", []) or []
    print(f"[after_model]   model returned {len(calls)} tool call(s)")
    return None


@after_agent
def log_after_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[after_agent]  run finished with {len(state['messages'])} message(s)")
    return None

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[log_before_agent, log_before_model, log_after_model, log_after_agent],
)

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]}
)

`before_agent` and `after_agent` fire once. `before_model` and `after_model` fire on
every model call — twice here, because the agent called a tool and then went back to
the model with the result.

## Wrap-style hooks

These sit *around* a call, so they see it start and finish, and can retry it, swap the
model, or change a tool result.

In [ ]:
from typing import Callable

from langchain.agents.middleware import (
    ModelRequest,
    ModelResponse,
    wrap_model_call,
    wrap_tool_call,
)


@wrap_model_call
def trace_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    print("[wrap_model_call] -> calling the model")
    response = handler(request)
    print("[wrap_model_call] <- model returned")
    return response


@wrap_tool_call
def trace_tool(request, handler):
    print(f"[wrap_tool_call]  {request.tool_call['name']}({request.tool_call['args']})")
    return handler(request)

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[log_before_agent, trace_model, log_after_model, trace_tool, log_after_agent],
)

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]}
)

## Human in the loop

`HumanInTheLoopMiddleware` pauses before a named tool runs and waits for a decision.
It needs a checkpointer, because pausing means saving state — the same `interrupt()`
mechanism from module-0.

`interrupt_on` names the tools to gate. Both tools are listed here — anything with
a side effect or a cost is worth gating.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"sql_query": True, "web_search": True},
        )
    ],
    checkpointer=InMemorySaver(),
)

In [ ]:
QUESTION = "Query the Chinook database to count the rows in the Artist table."

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [HumanMessage(content=QUESTION)]}, config=config)

print("paused:", "__interrupt__" in response)

In [ ]:
def pending(response):
    """The tool calls waiting on a decision."""
    return response["__interrupt__"][0].value["action_requests"]


for request in pending(response):
    print(request["name"], request["args"])

A model can ask for several tools in one turn, so `action_requests` is a list — and
you must return **exactly one decision per pending call**, or the middleware raises
`ValueError: Number of human decisions (n) does not match number of hanging tool
calls (m)`. Build the list from the requests rather than hard-coding it.

### Approve

Each approved tool runs with the arguments the model chose.

In [ ]:
response = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"} for _ in pending(response)]}),
    config=config,
)

print(response["messages"][-1].content)

### Reject

The tool does **not** run. Your reason goes back to the model as a tool message.

In [ ]:
config = {"configurable": {"thread_id": "2"}}
response = agent.invoke({"messages": [HumanMessage(content=QUESTION)]}, config=config)

response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "reject", "message": "Do not query the Artist table."}
                for _ in pending(response)
            ]
        }
    ),
    config=config,
)

print(response["messages"][-1].content)

In [ ]:
from langchain.messages import ToolMessage

for message in response["messages"]:
    if isinstance(message, ToolMessage):
        print(message.content)

### Edit

The tool runs with **your** arguments instead of the model's.

In [ ]:
config = {"configurable": {"thread_id": "3"}}
response = agent.invoke({"messages": [HumanMessage(content=QUESTION)]}, config=config)

edit = {
    "type": "edit",
    "edited_action": {
        "name": "sql_query",
        "args": {"query": "SELECT COUNT(*) FROM Album"},
    },
}

# edit the first pending call, approve any others
decisions = [edit] + [{"type": "approve"} for _ in pending(response)[1:]]

response = agent.invoke(Command(resume={"decisions": decisions}), config=config)

for message in response["messages"]:
    if isinstance(message, ToolMessage):
        print(message.content)

The edited arguments were used instead of the model's.

Whether the agent stops there or asks for another query is up to the model, and any
further tool call hits the same gate. In real code you keep resuming until the run
comes back without an interrupt:

In [ ]:
while "__interrupt__" in response:
    for request in pending(response):
        print("approving:", request["args"])
    response = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"} for _ in pending(response)]}),
        config=config,
    )

print(response["messages"][-1].content)